# Bronze Ingestion — Netflix Titles

Loads Netflix titles CSV files from the personal ADLS container into a Delta table in `ivanrazumovskyi_bronze`, using Auto Loader for incremental, idempotent loading.


## 1. Configuration

Defines all paths and table names in one place

In [0]:
storage_account = "dlspl21databricks"
container = "ivanrazumovskyi"
catalog = "dbr_dev"
bronze_schema = "ivanrazumovskyi_bronze"
target_table = "netflix_titles"

storage_root = f"abfss://{container}@{storage_account}.dfs.core.windows.net"

source_path = f"{storage_root}/ingestion/netflix/"
checkpoint_path = f"{storage_root}/checkpoints/netflix_titles/"
bronze_table = f"{catalog}.{bronze_schema}.{target_table}"

print("Source path:", source_path)
print("Checkpoint path:", checkpoint_path)
print("Target table:", bronze_table)

## 2. Verify source files are reachable

Sanity check that the external location gives read access to the landing folder before attempting any streaming read.

In [0]:
display(dbutils.fs.ls(source_path))

## 3. Define the explicit source schema


In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType
)

source_schema = StructType([
    StructField("show_id", StringType(), True),
    StructField("type", StringType(), True),
    StructField("title", StringType(), True),
    StructField("director", StringType(), True),
    StructField("cast", StringType(), True),
    StructField("country", StringType(), True),
    StructField("date_added", StringType(), True),
    StructField("release_year", IntegerType(), True),
    StructField("rating", StringType(), True),
    StructField("duration", StringType(), True),
    StructField("listed_in", StringType(), True),
    StructField("description", StringType(), True),
])

## 4. Read source files with Auto Loader and add metadata columns

In [0]:
from pyspark.sql.functions import col, current_timestamp, current_date

df_bronze = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .schema(source_schema)
    .load(source_path)
)

df_bronze = (
    df_bronze
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_load_date", current_date())
)

In [0]:
df_bronze.printSchema()

## 5. Write the stream to the Bronze Delta table

In [0]:
query = (
    df_bronze.writeStream
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

In [0]:
%sql
-- Row count check
SELECT COUNT(*) AS row_count FROM dbr_dev.ivanrazumovskyi_bronze.netflix_titles;

In [0]:
%sql
-- Rows per source file
SELECT _source_file, COUNT(*) AS row_count, MIN(_ingested_at) AS first_ingestion
FROM dbr_dev.ivanrazumovskyi_bronze.netflix_titles
GROUP BY _source_file
ORDER BY _source_file;

## 6. Idempotency test

In [0]:
count_before = spark.sql(f"SELECT COUNT(*) AS c FROM {bronze_table}").collect()[0]["c"]
print(f"Row count BEFORE re-run: {count_before}")

In [0]:
query = (
    df_bronze.writeStream
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

In [0]:
count_after = spark.sql(f"SELECT COUNT(*) AS c FROM {bronze_table}").collect()[0]["c"]
print(f"Row count AFTER re-run: {count_after}")

if count_before == count_after:
    print("✅ IDEMPOTENCY TEST PASSED — no duplicate rows after re-running with no new files")
else:
    print(f"❌ IDEMPOTENCY TEST FAILED — row count changed by {count_after - count_before}")

## 7. Incremental load test

In [0]:
count_before_new_file = spark.sql(f"SELECT COUNT(*) AS c FROM {bronze_table}").collect()[0]["c"]
print(f"Row count BEFORE adding new file: {count_before_new_file}")

In [0]:
query = (
    df_bronze.writeStream
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

In [0]:
count_after_new_file = spark.sql(f"SELECT COUNT(*) AS c FROM {bronze_table}").collect()[0]["c"]
new_rows = count_after_new_file - count_before_new_file

print(f"Row count AFTER adding new file: {count_after_new_file}")
print(f"New rows added: {new_rows}")

if new_rows > 0:
    print("✅ NEW FILE TEST PASSED — only the new file's rows were added")
else:
    print("❌ NEW FILE TEST FAILED — no new rows were added")

## 8. Final per-file breakdown

In [0]:
%sql
SELECT
    regexp_extract(_source_file, '[^/]+$', 0) AS source_file,
    COUNT(*) AS row_count,
    MIN(_ingested_at) AS first_ingestion,
    MAX(_ingested_at) AS last_ingestion
FROM dbr_dev.ivanrazumovskyi_bronze.netflix_titles
GROUP BY source_file
ORDER BY source_file;